In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define SPY and VIX price grids
spy_prices = np.linspace(580, 620, 100)
vix_prices = np.linspace(14, 20, 100)
spy_grid, vix_grid = np.meshgrid(spy_prices, vix_prices)

# Premiums
spy_premium = 1.00  # example premium
vix_premium = 1.00  # example premium

# VIX X/X put spread payoff
def vix_putspread(vix):
    return np.maximum(18 - vix, 0)

# SPY butterfly: long XP, short Xx XP, long XP
def spy_butterfly(spy):
    leg1 = np.maximum(620 - spy, 0)
    leg2 = -2 * np.maximum(624 - spy, 0)
    leg3 = np.maximum(628 - spy, 0)
    return leg1 + leg2 + leg3

# Compute payoffs
spy_pnl = spy_butterfly(spy_grid) - spy_premium
vix_pnl = vix_putspread(vix_grid) - vix_premium
total_pnl = spy_pnl + vix_pnl

# Plot the XD surface
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(spy_grid, vix_grid, total_pnl, cmap='RdYlGn', edgecolor='none')

# Labels and title
ax.set_xlabel('SPY Price at Expiration')
ax.set_ylabel('VIX Price at Expiration')
ax.set_zlabel('Total PnL ($)')
ax.set_title('3D PnL Surface: SPY 620/624/628 Butterfly + VIX 18 Put')

# Plot actual outcome (July X, X)
actual_spy = 600.0
actual_vix = 16.0
actual_spy_pnl = spy_butterfly(actual_spy) - spy_premium
actual_vix_pnl = vix_putspread(actual_vix) - vix_premium
actual_total_pnl = actual_spy_pnl + actual_vix_pnl
print(actual_total_pnl)

ax.scatter(actual_spy, actual_vix, actual_total_pnl, color='black', s=20, label='Actual Outcome')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

def compute_performance_metrics(returns, periods_per_year=252, risk_free_rate=0.0):
    """
    Compute Sharpe Ratio, Sortino Ratio, Max Drawdown, and Winrate from a return series.

    Parameters:
        returns (list or pd.Series): Periodic returns (e.g., daily or weekly)
        periods_per_year (int): Number of periods in a year (252 for daily, 52 for weekly)
        risk_free_rate (float): Optional risk-free rate per period (not annualized)

    Returns:
        dict: Dictionary with keys 'Sharpe', 'Sortino', 'Max Drawdown', 'Winrate'
    """
    returns = pd.Series(returns).dropna()

    # Excess returns
    excess_returns = returns - risk_free_rate

    # Sharpe Ratio
    sharpe = excess_returns.mean() / excess_returns.std() * np.sqrt(periods_per_year)

    # Sortino Ratio (downside deviation)
    downside = excess_returns[excess_returns < 0]
    sortino = excess_returns.mean() / downside.std() * np.sqrt(periods_per_year)

    # Max Drawdown
    cumulative = (1 + returns).cumprod()
    peak = cumulative.cummax()
    drawdown = cumulative / peak - 1
    max_drawdown = drawdown.min()

    # Winrate
    winrate = (returns > 0).mean() * 100

    return {
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max Drawdown': max_drawdown,
        'Winrate': winrate
    }
